# DeepForge — real NumPy neural-network training

Train a small nonlinear binary classifier entirely in NumPy and watch DeepForge's inline dashboard during training. The dataset is synthetic and the recorded loss/accuracy are computed from real forward/backward updates, not hard-coded metrics. This is a CPU-friendly functional demonstration, not a benchmark.

Works with the Python notebook API; intended for Colab, VS Code and local Jupyter. No Tk window is required. In a local graphical environment the same snapshot can also be opened in the native read-only monitor.

## 1. Install and verify

Run once in your notebook kernel if dependencies are missing:

```python
%pip install "deepforge-research-toolkit[notebook,resources]==0.7.1.1.3" numpy
```

If that release is not yet available from PyPI, follow the repository README to install the matching source checkout. Restart an already-imported kernel after upgrading. The notebook assumes installation is complete before the next cell.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from uuid import uuid4
import numpy as np
from deepforge.monitor import TrainingMonitor
from deepforge.notebook import NotebookMonitor, snapshot_html

try:
    print("DeepForge:", version("deepforge-research-toolkit"))
except PackageNotFoundError:
    print("DeepForge imported from a source checkout")
print("NumPy:", np.__version__)


In [ ]:
# Independent, deterministic train and validation samples.
rng = np.random.default_rng(42)
def make_data(n):
    x = rng.normal(size=(n, 2))
    y = (x[:, 0] * x[:, 1] > 0).astype(float)
    return x, y

X, Y = make_data(512)
X_val, Y_val = make_data(128)
EPOCHS, BATCH_SIZE, LR = 20, 32, 0.25
W1 = rng.normal(0, 0.5, (2, 16))
b1 = np.zeros(16)
W2 = rng.normal(0, 0.25, (16, 1))
b2 = np.zeros(1)

def forward(x):
    h = np.tanh(x @ W1 + b1)
    z = (h @ W2 + b2).ravel()
    p = 1 / (1 + np.exp(-np.clip(z, -50, 50)))
    return h, z, p

def metrics(x, y):
    _, z, p = forward(x)
    return float(np.mean(np.logaddexp(0, z) - y*z)), float(100*np.mean((p >= .5) == y))

def train_batch(x, y):
    global W1, b1, W2, b2
    h, _, p = forward(x)
    dz = ((p-y)/len(x))[:, None]
    dw2, db2 = h.T @ dz, dz.sum(axis=0)
    dh = (dz @ W2.T) * (1-h*h)
    dw1, db1 = x.T @ dh, dh.sum(axis=0)
    grad_norm = float(np.sqrt(sum(float(np.square(g).sum()) for g in (dw1, db1, dw2, db2))))
    W1 -= LR*dw1; b1 -= LR*db1
    W2 -= LR*dw2; b2 -= LR*db2
    return grad_norm

baseline = metrics(X_val, Y_val)
print("Initial validation loss, accuracy:", baseline)


In [ ]:
# A new output directory for every independent experiment.
run_dir = Path("deepforge_demo_runs") / ("numpy_" + uuid4().hex[:10])
run_dir.mkdir(parents=True)
path = run_dir / "live.json"
batches = len(X) // BATCH_SIZE
monitor = TrainingMonitor(path, epochs=EPOCHS, batches_per_epoch=batches,
                          publish_interval=0, resource_interval=1)
view = NotebookMonitor(path, interval=0.5, max_epochs=20, theme="jade",
                       fixed_header=True, dashboard_height=650)
view.refresh(force=True)
print("Snapshot:", path.resolve())


In [ ]:
# Real optimizer updates, measured validation, and genuine inline monitoring.
try:
    for epoch in range(1, EPOCHS + 1):
        order = rng.permutation(len(X))
        for batch, offset in enumerate(range(0, len(X), BATCH_SIZE), 1):
            idx = order[offset:offset+BATCH_SIZE]
            grad_norm = train_batch(X[idx], Y[idx])
            # Report current full-training metrics for a consistent per-epoch curve.
            loss, accuracy = metrics(X, Y)
            monitor.batch(epoch=epoch, batch=batch, samples=len(idx),
                          metrics={"loss": loss, "accuracy": accuracy, "lr": LR,
                                   "gradient_norm": grad_norm}, labels=Y[idx].astype(int).tolist())
            view.refresh()
            if epoch == 8 and batch == 8:
                view.refresh(force=True)
                (run_dir / "during-training.html").write_text(
                    snapshot_html(path, dashboard_height=650), encoding="utf-8")
        monitor.record_stage(epoch, "validation", "running", completed=0, total=1)
        val_loss, val_accuracy = metrics(X_val, Y_val)
        monitor.epoch_end(epoch, {"loss": val_loss, "accuracy": val_accuracy})
        monitor.record_stage(epoch, "validation", "completed", completed=1, total=1)
        monitor.record_stage(epoch, "checkpoint", "skipped")
        monitor.record_stage(epoch, "verification", "skipped")
        view.refresh(force=True)
    monitor.finish()
except Exception:
    monitor.finish(failed=True)
    raise
finally:
    view.refresh(force=True)


In [ ]:
assert monitor.snapshot()["status"] == "completed"
assert len(monitor.snapshot()["epoch_history"]) == EPOCHS
assert monitor.snapshot()["completed_batches"] == EPOCHS*batches
final = metrics(X_val, Y_val)
assert np.isfinite(final).all()
assert final[0] < baseline[0]
(run_dir / "completed.html").write_text(snapshot_html(path, dashboard_height=650), encoding="utf-8")
print("Initial validation:", baseline)
print("Final validation:", final)
print("Artifacts:", run_dir.resolve())


## 5. Optional local desktop viewer

If you run this notebook on a local machine with Tkinter, you can open the same snapshot in a separate terminal (replace PATH with the printed absolute path):

```bash
deepforge-monitor "PATH_TO_RUN/live.json"
```

For the legacy viewer use `deepforge-live "PATH_TO_RUN/live.json"`. Launch during training to see a live read-only view, or after training to inspect the saved snapshot. Hosted Colab/Kaggle cannot open a desktop Tk window. The NumPy demo intentionally does not save training weights; for managed checkpoint/resume use the PyTorch session notebook. Every real training metric comes from the caller's loop.